In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Estimator 빠른 시작

Estimator Primitive는 양자 Circuit으로 준비된 상태에 대해 하나 이상의 Observable의 기댓값을 계산합니다. Circuit은 매개변수화될 수 있으며, 이 경우 매개변수 값도 Primitive에 입력으로 제공해야 합니다.

이 Primitive에는 동적 분리, Pauli 트월링, 게이트 폴딩 ZNE, PEA, PEC를 포함한 여러 내장 [오류 완화 및 억제 기법](/guides/error-mitigation-and-suppression-techniques)이 있습니다. 또한 비용과 정확도 트레이드오프를 쉽게 구성할 수 있는 `resilience_level` 옵션을 지원합니다.
이 주제의 단계는 Estimator를 설정하고, 구성에 사용할 수 있는 옵션을 탐색하고, 프로그램에서 호출하는 방법을 설명합니다.


<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

이는 실험적 기능으로 향후 변경될 수 있습니다.

:::

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

print(backend.name)

ibm_fez


### 2. Create a circuit and an observable

You need at least one circuit and one observable as inputs to the Estimator primitive.

In [2]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

entanglement = [tuple(edge) for edge in backend.coupling_map.get_edges()]
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 0.5) for i, j in entanglement],
    num_qubits=backend.num_qubits,
)
circuit = qaoa_ansatz(observable, reps=2)
# The circuit is parametrized, so we will define the parameter values for execution
param_values = [0.1, 0.2, 0.3, 0.4]

### 2. Circuit과 Observable 생성
Estimator Primitive에 대한 입력으로 최소 하나의 Circuit과 하나의 Observable이 필요합니다.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 4472), ('sx', 1884), ('cz', 1120)])


Circuit과 Observable은 QPU에서 지원되는 명령만 사용하도록 변환되어야 합니다(*명령 집합 아키텍처(ISA)* Circuit이라고 함). 이를 위해 트랜스파일러를 사용하세요.

In [4]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

### 4. Invoke Estimator and get results

Next, invoke the `run()` method to calculate expectation values for the input circuits and observables. The circuit, observable, and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = estimator.run([(isa_circuit, isa_observable, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d82869ntjchs73bnokog


>>> Job Status: QUEUED


In [6]:
result = job.result()
print(f">>> {result}")
print(f"  > Expectation value: {result[0].data.evs}")
print(f"  > Metadata: {result[0].metadata}")

>>> PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})
  > Expectation value: 30.60337496305257
  > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


### 4. Estimator 호출 및 결과 가져오기
다음으로, `run()` 메서드를 호출하여 입력 Circuit과 Observable의 기댓값을 계산합니다. Circuit, Observable, 선택적 매개변수 값 집합은 *Primitive Unified Bloc* (PUB) 튜플로 입력됩니다.